<a href="https://colab.research.google.com/github/kshitijagame-tech/rabtech-aiml-customer-churn/blob/main/Task_4_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report
)

import joblib

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
import seaborn as sns

# Load dataset
df = sns.load_dataset("titanic")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully!
Shape: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [ ]:
# Remove columns that directly reveal the target
df = df.drop(columns=["alive", "class", "deck"], errors="ignore")

# Target variable
y = df["survived"]

# Input features
X = df.drop(columns=["survived"])

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

Features shape: (891, 11)
Target shape: (891,)

Target distribution:
survived
0    549
1    342
Name: count, dtype: int64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (712, 11)
Testing data: (179, 11)


In [ ]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['pclass', 'age', 'sibsp', 'parch', 'fare']

Categorical features:
['sex', 'embarked', 'who', 'adult_male', 'embark_town', 'alone']


In [ ]:
# Numerical preprocessing
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both pipelines
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

print("4 ML models created successfully!")

for name in models:
    print("-", name)

4 ML models created successfully!
- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Stratified 5-Fold Cross-Validation created successfully!")

Stratified 5-Fold Cross-Validation created successfully!


In [ ]:
param_grids = {
    "Logistic Regression": {
        "model__C": [0.1, 1, 10]
    },

    "Decision Tree": {
        "model__max_depth": [3, 5, 7],
        "model__min_samples_split": [2, 5]
    },

    "Random Forest": {
        "model__n_estimators": [50, 100],
        "model__max_depth": [5, 10, None]
    },

    "Gradient Boosting": {
        "model__n_estimators": [50, 100],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [3, 5]
    }
}

print("Hyperparameter grids created successfully!")

Hyperparameter grids created successfully!


In [ ]:
results = []
best_models = {}

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[name],
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    best_models[name] = grid_search.best_estimator_

    print("\n" + name)
    print("Best Parameters:", grid_search.best_params_)
    print("Best CV ROC-AUC:", round(grid_search.best_score_, 4))

print("\nAll 4 models tuned successfully!")


Logistic Regression
Best Parameters: {'model__C': 0.1}
Best CV ROC-AUC: 0.8641

Decision Tree
Best Parameters: {'model__max_depth': 3, 'model__min_samples_split': 2}
Best CV ROC-AUC: 0.8568

Random Forest
Best Parameters: {'model__max_depth': 5, 'model__n_estimators': 50}
Best CV ROC-AUC: 0.8717

Gradient Boosting
Best Parameters: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100}
Best CV ROC-AUC: 0.8925

All 4 models tuned successfully!


In [ ]:
evaluation_results = []

for name, model in best_models.items():

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    evaluation_results.append({
        "Model": name,
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(evaluation_results)

results_df = results_df.sort_values(
    by="ROC-AUC",
    ascending=False
).reset_index(drop=True)

results_df

,Model,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.803279,0.710145,0.753846,0.868511
1,Random Forest,0.819672,0.724638,0.769231,0.853030
2,Decision Tree,0.787879,0.753623,0.770370,0.850593
3,Gradient Boosting,0.807018,0.666667,0.730159,0.849078


In [ ]:
for name, model in best_models.items():

    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    print("\n" + name)
    print("Confusion Matrix:")
    print(cm)


Logistic Regression
Confusion Matrix:
[[98 12]
 [20 49]]

Decision Tree
Confusion Matrix:
[[96 14]
 [17 52]]

Random Forest
Confusion Matrix:
[[99 11]
 [19 50]]

Gradient Boosting
Confusion Matrix:
[[99 11]
 [23 46]]


In [14]:
best_model_name = results_df.iloc[0]["Model"]
best_model = best_models[best_model_name]

print("🏆 CHAMPION MODEL")
print("Model:", best_model_name)
print("ROC-AUC:", round(results_df.iloc[0]["ROC-AUC"], 4))
print("Precision:", round(results_df.iloc[0]["Precision"], 4))
print("Recall:", round(results_df.iloc[0]["Recall"], 4))
print("F1 Score:", round(results_df.iloc[0]["F1 Score"], 4))

🏆 CHAMPION MODEL
Model: Logistic Regression
ROC-AUC: 0.8685
Precision: 0.8033
Recall: 0.7101
F1 Score: 0.7538


In [15]:
model_filename = "/content/Task_4_Champion_Model.joblib"

joblib.dump(best_model, model_filename)

print("Champion model saved successfully!")
print("File:", model_filename)

import os
print("File exists:", os.path.exists(model_filename))

Champion model saved successfully!
File: /content/Task_4_Champion_Model.joblib
File exists: True


In [16]:
loaded_model = joblib.load(model_filename)

loaded_predictions = loaded_model.predict(X_test)

print("Saved model loaded successfully!")
print("Number of test samples:", len(X_test))
print("Number of predictions:", len(loaded_predictions))
print("Model verification successful!")

Saved model loaded successfully!
Number of test samples: 179
Number of predictions: 179
Model verification successful!


In [17]:
print("========== FINAL MODEL COMPARISON ==========")

display(
    results_df.style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1 Score": "{:.4f}",
        "ROC-AUC": "{:.4f}"
    })
)

print("\n🏆 Champion Model:", best_model_name)
print("Selected based on highest ROC-AUC score.")

========== FINAL MODEL COMPARISON ==========


,Model,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.8033,0.7101,0.7538,0.8685
1,Random Forest,0.8197,0.7246,0.7692,0.8530
2,Decision Tree,0.7879,0.7536,0.7704,0.8506
3,Gradient Boosting,0.8070,0.6667,0.7302,0.8491



🏆 Champion Model: Logistic Regression
Selected based on highest ROC-AUC score.


In [18]:
y_pred_best = best_model.predict(X_test)

print("========== CHAMPION MODEL CLASSIFICATION REPORT ==========")
print("Champion Model:", best_model_name)
print()

print(classification_report(y_test, y_pred_best))

========== CHAMPION MODEL CLASSIFICATION REPORT ==========
Champion Model: Logistic Regression

              precision    recall  f1-score   support

           0       0.83      0.89      0.86       110
           1       0.80      0.71      0.75        69

    accuracy                           0.82       179
   macro avg       0.82      0.80      0.81       179
weighted avg       0.82      0.82      0.82       179



In [19]:
print("========== TASK 4 SUMMARY ==========")
print("Dataset: Titanic")
print("Task Type: Supervised Classification")
print("Models Tested: 4")
print("Models: Logistic Regression, Decision Tree, Random Forest, Gradient Boosting")
print("Hyperparameter Tuning: GridSearchCV")
print("Cross-Validation: Stratified 5-Fold")
print("Evaluation Metrics: Precision, Recall, F1 Score, ROC-AUC")
print("Champion Model:", best_model_name)
print("Model Saved As:", model_filename)
print("Data Leakage: Prevented using Pipeline and Cross-Validation")
print("=====================================")

========== TASK 4 SUMMARY ==========
Dataset: Titanic
Task Type: Supervised Classification
Models Tested: 4
Models: Logistic Regression, Decision Tree, Random Forest, Gradient Boosting
Hyperparameter Tuning: GridSearchCV
Cross-Validation: Stratified 5-Fold
Evaluation Metrics: Precision, Recall, F1 Score, ROC-AUC
Champion Model: Logistic Regression
Model Saved As: /content/Task_4_Champion_Model.joblib
Data Leakage: Prevented using Pipeline and Cross-Validation
